In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_TRACING_V2'] = "true" 
os.environ['LANGCHAIN_PROJECT'] = os.getenv("LANGCHAIN_PROJECT")

### Steps for Simple GEN AI App
1. Load Data to Docs
2. Divide text into chunks of text
3. Convert text to vectors using Vector Embeddings
4. Store in Vector store DB

##### Data injection from Website & Storing it into vector store db (FAISS)

In [28]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaLLM
from langchain_classic.chains import create_retrieval_chain
from langchain_core.documents import Document

In [29]:
loader = WebBaseLoader("https://docs.langchain.com/langsmith/observability-concepts")

In [30]:
docs = loader.load()

In [31]:
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-concepts', 'title': 'Observability concepts - Docs by LangChain', 'language': 'en'}, page_content='Observability concepts - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationObservability conceptsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsTutorialPolly AI assistantTracing setupIntegrationsManual instrumentationThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroubleshooting guidesViewing & managing tracesFilter tracesConfigure run previewsManage a traceQuery tracesSDKQuery threadsSDKLangSmith MCP ServerBulk export trace dataAutomationsSet up automation rulesConfigure webhook notifica

In [32]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents = splitter.split_documents(docs)

In [33]:
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-concepts', 'title': 'Observability concepts - Docs by LangChain', 'language': 'en'}, page_content='Observability concepts - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationObservability conceptsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsTutorialPolly AI assistantTracing setupIntegrationsManual instrumentationThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroubleshooting guidesViewing & managing tracesFilter tracesConfigure run previewsManage a traceQuery tracesSDKQuery threadsSDKLangSmith MCP ServerBulk export trace dataAutomationsSet up automation rulesConfigure webhook notifica

In [34]:
embeddings = OllamaEmbeddings(model='gemma:2b')

In [35]:
db = FAISS.from_documents(documents, embeddings)

### Retrivers & Chains

In [37]:
query = "LangSmith Observability lets you record, inspect, and analyze"

result = db.similarity_search(query)

result[0].page_content

'Browse all integrations.\n\u200bManual instrumentation\nManual instrumentation lets you add tracing to any code, regardless of the framework. Use it when you’re not using a supported integration or when you need granular control over what gets traced. LangSmith provides three mechanisms:'

In [38]:
llm = OllamaLLM(model="gemma:2b")

prompt = ChatPromptTemplate.from_template(
    """
    Answer the following question based only on provided context:
    <context>
    {context}
    </context>
    """
)

document_chain = create_stuff_documents_chain(llm, prompt)

document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on provided context:\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
| OllamaLLM(model='gemma:2b')
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [46]:

document_chain.invoke({
    "input": "LangSmith Observability lets you record, inspect, and analyze",
    "context": [Document(page_content="A trace is a collection of runs for a single operation. For example, if a user request triggers a chain that calls an LLM and then an output parser, all of those runs belong to the same trace. Runs are bound to a trace by a unique trace ID. If you are familiar with OpenTelemetry, you can think of a LangSmith trace as a collection of spans.")]
})

'Sure, I understand the context and can answer the question based on it.\n\n**Answer:**\n\nA trace is a collection of runs for a single operation, where each run is bound to a unique trace ID.'

In [47]:
retriver = db.as_retriever()

In [48]:
retriver_chain = create_retrieval_chain(retriver, document_chain)

In [49]:
retriver_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000024F21DAF3E0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on provided context:\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
  

In [50]:
response = retriver_chain.invoke({
    "input": "LangSmith Observability lets you record, inspect, and analyze"
})

response

{'input': 'LangSmith Observability lets you record, inspect, and analyze',
 'context': [Document(id='4ccca5c6-e030-4daa-bfd9-82c509824da8', metadata={'source': 'https://docs.langchain.com/langsmith/observability-concepts', 'title': 'Observability concepts - Docs by LangChain', 'language': 'en'}, page_content='Browse all integrations.\n\u200bManual instrumentation\nManual instrumentation lets you add tracing to any code, regardless of the framework. Use it when you’re not using a supported integration or when you need granular control over what gets traced. LangSmith provides three mechanisms:'),
  Document(id='d735575d-eb80-437c-b936-619a8a495e86', metadata={'source': 'https://docs.langchain.com/langsmith/observability-concepts', 'title': 'Observability concepts - Docs by LangChain', 'language': 'en'}, page_content='\u200bHow LangSmith structures data\nLangSmith groups multiple traces within a project. Each trace records the sequence of steps your application takes for a single operati